In [5]:
import sys
print(sys.prefix)

C:\Users\mosab\anaconda3\envs\tf


In [6]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Check if a GPU is available and set the device
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("bigscience/bloomz-7b1")

# Load the model and quantize it to 4-bit
model = AutoModelForCausalLM.from_pretrained("bigscience/bloomz-7b1", 
                                             #device_map='auto', 
                                             load_in_8bit=True,
                                             torch_dtype=torch.float16,
                                             #bnb_4bit_compute_dtype=torch.float16  # Set the compute data type to float16 for better performance
                                            )#.to(device)

# Load the ARCD dataset from CSV
df = pd.read_csv(r'C:\Users\mosab\Desktop\Datasets\General\ARCD\arcd_all.csv', encoding='utf-16', sep='\t')

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
`low_cpu_mem_usage` was None, now set to True since model is quantized.


In [7]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import numpy as np
import re

# Text normalization for comparison (lowercase, remove articles, punctuation, etc.)
def normalize_text(text):
    text = text.lower()
    #text = re.sub(r'\b(a|an|the)\b', ' ', text)  # Remove articles
    text = re.sub(r'[\u064B-\u065F]', '', text)  # Remove diacritics
    text = re.sub(r'\W+', ' ', text)  # Remove punctuation and double whitespaces
    #text = re.sub(r'[^a-zء-ي0-9]+', ' ', text)  # Remove double whitespaces
    #text = ' '.join(text.split())  # Remove extra spaces
    return text

# Exact Match (EM) calculation
def exact_match_score(predicted_answer, actual_answer):
    return int(normalize_text(predicted_answer) == normalize_text(actual_answer))


# F1 Score calculation for a single prediction
def f1_single(predicted_answer, actual_answer):
    # Tokenize predicted and actual answers
    pred_tokens = normalize_text(predicted_answer).split()
    actual_tokens = normalize_text(actual_answer).split()
    
    if (len(pred_tokens) == 0) and (len(actual_tokens) == 0):
        return 1
    elif (len(pred_tokens) == 0):
        return 0
    elif (len(actual_tokens) == 0):
        return 0
     #common_tokens = pred_tokens.intersection(actual_tokens)
    common_tokens = set(pred_tokens) & set(actual_tokens)
    #common_tokens = remove_stopwords_from_set(common_tokens, stopwords_list)
    if len(common_tokens) == 0:
        return 0.0
    
    precision = len(common_tokens) / len(pred_tokens)
    recall = len(common_tokens) / len(actual_tokens)
    f1 = 2 * (precision * recall) / (precision + recall)
    return f1

# Function to calculate average EM and F1 scores across the dataset
def evaluate_qa(predicted_answers, actual_answers):
    total_em = 0
    total_f1 = 0
    n = len(predicted_answers)
    
    for predicted, actual in zip(predicted_answers, actual_answers):
        total_em += exact_match_score(predicted, actual)
        total_f1 += f1_single(predicted, actual)
    
    # Calculate the average EM and F1 scores
    em_score = total_em / n
    f1_score_avg = total_f1 / n
    return em_score, f1_score_avg


# Function to compute exact match (EM) and F1 score
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    # Strip white space in predictions and labels
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]
    
    exact_match_score, avg_f1_score = evaluate_qa(decoded_preds, decoded_labels)
    
    # Exact Match
    #exact_matches = [1 if pred == label else 0 for pred, label in zip(decoded_preds, decoded_labels)]
    #exact_match_score = sum(exact_matches) / len(exact_matches)
    
    # F1 Score
    #f1_scores = [f1_metric.compute(predictions=[pred], references=[label])['f1'] for pred, label in zip(decoded_preds, decoded_labels)]
    #avg_f1_score = sum(f1_scores) / len(f1_scores)
    
    
    return {
        "exact_match": exact_match_score,
        "f1": avg_f1_score
    }


In [10]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from transformers import BitsAndBytesConfig
#import evaluate

# 1. Prepare the dataset (Assumes the CSV has context, title, question, and answers)
df = pd.read_csv(r'C:\Users\mosab\Desktop\Datasets\General\ARCD\arcd_all.csv', encoding='utf-16', sep='\t')
df_train = df[0:int(df.shape[0]*0.8)]
df_val = df[int(df.shape[0]*0.8)+1:int(df.shape[0]*0.9)]
df_test = df[int(df.shape[0]*0.9)+1:df.shape[0]]#-1

# 2. Function to create prompt (make sure this returns a dictionary)
def create_prompt(row):
    title = row['title']
    context = row['context']
    question = row['question']
    context=f"\nTitle: {title}\n"+context
    # Combine title, context, and question to create the prompt
    prompt = f"Context: {context}\n\nQuestion: {question}\nInstruction: When there is no answer in the context, don't output any word.\nAnswer:"
    #prompt = f"Title: {title}\nContext: {context}\nQuestion: {question}\nAnswer:"
    return {'input': prompt, 'output': row['answer']}  # Return dictionary

# 3. Apply the prompt creation function to each dataset
train_prompts = df_train.apply(create_prompt, axis=1).tolist()  # Convert to list of dictionaries
val_prompts = df_val.apply(create_prompt, axis=1).tolist()
test_prompts = df_test.apply(create_prompt, axis=1).tolist()

# 4. Convert to pandas DataFrame to ensure compatibility with Dataset
train_df = pd.DataFrame(train_prompts)
val_df = pd.DataFrame(val_prompts)
test_df = pd.DataFrame(test_prompts)

# 5. Convert the DataFrame to HuggingFace Dataset
train_hf_dataset = Dataset.from_pandas(train_df)
val_hf_dataset = Dataset.from_pandas(val_df)
test_hf_dataset = Dataset.from_pandas(test_df)



In [ ]:
from datetime import datetime
now = datetime.now()
print(now.time())

In [ ]:

# 5. Load the tokenizer and the model with 4-bit precision
#model_name = "bigscience/bloomz-7b1"

# Enable 4-bit precision to reduce memory usage
#bnb_config = BitsAndBytesConfig(load_in_4bit=True)

#tokenizer = AutoTokenizer.from_pretrained(model_name)
#model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config)

# 6. Tokenization function
def tokenize_function(examples):
    inputs = tokenizer(examples['input'], padding="longest", truncation=False, return_tensors="pt")
    #inputs = tokenizer(examples['input'], padding="max_length", truncation=True, max_length=2400, return_tensors="pt")
    labels = tokenizer(examples['output'], padding="longest", truncation=True, return_tensors="pt").input_ids
    #labels = tokenizer(examples['output'], padding="max_length", truncation=True, max_length=200, return_tensors="pt").input_ids
    inputs['labels'] = labels
    return inputs

# 7. Tokenize the datasets
train_tokenized = train_hf_dataset.map(tokenize_function, batched=True, remove_columns=["input", "output"])
val_tokenized = val_hf_dataset.map(tokenize_function, batched=True, remove_columns=["input", "output"])
test_tokenized = test_hf_dataset.map(tokenize_function, batched=True, remove_columns=["input", "output"])

# 8. Define TrainingArguments
training_args = TrainingArguments(
    output_dir="./bloomz_finetuned_qa_4bit",
    eval_strategy="steps",
    learning_rate=2e-5,
    per_device_train_batch_size=2,  # Adjust based on your GPU memory
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    save_steps=10_000,
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=500,
    fp16=True  # Mixed precision training for more efficiency
)

# 9. Exact Match and F1 Score Metrics
# Load evaluation metrics
#em_metric = evaluate.load("exact_match")
#f1_metric = evaluate.load("f1")

# 10. Define the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
)

# 11. Train the model
trainer.train()

# 12. Evaluate on validation and test datasets
print("Evaluating on Validation Set")
validation_results = trainer.evaluate(eval_dataset=val_tokenized)
print(f"Validation Results: {validation_results}")

print("Evaluating on Test Set")
test_results = trainer.evaluate(eval_dataset=test_tokenized)
print(f"Test Results: {test_results}")


In [12]:
now = datetime.now()
print(now.time())

12:19:14.815110


In [ ]:
model_path = r'C:\Users\mosab\Desktop\Jupyter Notebook\Research\models\'

# Save the entire model (architecture + weights + optimizer state)
model.save(model_path + "bloomz-7b1 - model - standard training")
tokenizer.save_pretrained(model_path + "bloomz-7b1 - tokenizer - standard training")